# 03 — Label the Merged Dataset

**Function:** Apply terrain/surface labels to the synchronised and resampled sensor data from `merged_dataset.csv`.

## Strategy

1. Load `merged_dataset_{dataset}.csv` from `data/interim/merged/`
2. Split by `run_id` to process each run individually
3. Discover label configurations from raw data directories
4. Apply labels to each run's sensor data
5. Save all labelled data to a single `labeled_dataset.csv` file
6. Generate labelled sensor plots for each run

**Input:** `data/interim/merged/merged_dataset_*.csv`
**Output:** `data/interim/labeled/labeled_dataset_*.csv` and per-run plots in `reports/labeled/`.


## Setup


In [1]:
from pathlib import Path
import sys
import pandas as pd

# Make project root importable whether CWD is repo root or notebooks/
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / 'src').exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.labels import (
    load_label_config, 
    discover_labeled_runs, 
    label_dataframe,
    validate_label_transitions, 
    plot_labeled_sensors, 
    plot_labeled_acc, 
    plot_labeled_gyro, 
    plot_labeled_odo, 
    save_labeled_plot
)


## Load the merged dataset

Load the synchronised, resampled run table produced by notebook 02.


In [2]:
# Load merged dataset from resampled data
merged_dataset_path = PROJECT_ROOT / "data" / "interim" / "merged" / "merged_dataset_Farm.csv"

#output file name
#output_file = "labeled_dataset.csv" --- IGNORE --- this was for the Main dataset
output_file = "labeled_dataset_Farm.csv"

if not merged_dataset_path.exists():
    raise FileNotFoundError(f"Merged dataset not found: {merged_dataset_path}\nPlease run 02_sync_resample.ipynb first.")

merged_df = pd.read_csv(merged_dataset_path)

print(f"Loaded merged dataset: {merged_dataset_path}")
print(f"Shape: {merged_df.shape}")
print(f"Columns: {list(merged_df.columns)}")
print(f"Unique run_ids: {merged_df['run_id'].nunique()}")
print(f"\nRun IDs:")
for run_id in sorted(merged_df['run_id'].unique()):
    n_samples = len(merged_df[merged_df['run_id'] == run_id])
    print(f"  - {run_id}: {n_samples} samples")


Loaded merged dataset: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/data/interim/merged/merged_dataset_Farm.csv
Shape: (467313, 11)
Columns: ['t', 't_rel', 'run_id', 'ax', 'ay', 'az', 'gx', 'gy', 'gz', 'v1', 'v2']
Unique run_ids: 5

Run IDs:
  - Run1: 52204 samples
  - Run2: 90033 samples
  - Run3: 80944 samples
  - Run4: 80700 samples
  - Run5: 163432 samples


## Discover label configs

Find every `labels_config.json` under `data/raw/` and pair it with its corresponding `run_id`.


In [3]:
# Discover label configurations and match with runs in merged dataset
raw_data_dir = PROJECT_ROOT / "data" / "raw"
runs_with_labels = discover_labeled_runs(raw_data_dir)

print(f"Found {len(runs_with_labels)} run(s) with label config(s):")
for run_dir, config_path in runs_with_labels.items():
    print(f"  - {run_dir.name} -> {config_path.relative_to(PROJECT_ROOT)}")

# Split merged dataset by run_id and match with label configs
run_data_dict = {}
for run_id in merged_df['run_id'].unique():
    # Find label config for this run_id
    label_config_path = None
    for run_dir, config_path in runs_with_labels.items():
        if run_dir.name == run_id:
            label_config_path = config_path
            break
    
    if label_config_path:
        run_subset = merged_df[merged_df['run_id'] == run_id].copy().reset_index(drop=True)
        label_configs = load_label_config(label_config_path)
        
        run_data_dict[run_id] = {
            'data': run_subset,
            'label_configs': label_configs,
            'config_path': label_config_path
        }
        
        print(f"\n{run_id}:")
        print(f"  Label config: {label_config_path.relative_to(PROJECT_ROOT)}")
        print(f"  Labels: {', '.join(label_configs.keys())}")
        print(f"  Samples: {len(run_subset)}")
    else:
        print(f"\n⚠ Warning: No label config found for {run_id}")

print(f"\nTotal runs with labels: {len(run_data_dict)}")


Found 9 run(s) with label config(s):
  - log_20260223_142511.490 -> data/raw/Main/log_20260223_142511.490/labels_config.json
  - log_20260326_120021.508 -> data/raw/Main/log_20260326_120021.508/labels_config.json
  - log_20260226_102148.990 -> data/raw/Main/log_20260226_102148.990/labels_config.json
  - log_20260309_141435.414 -> data/raw/Main/log_20260309_141435.414/labels_config.json
  - Run2 -> data/raw/Farm/Run2/labels_config.json
  - Run5 -> data/raw/Farm/Run5/labels_config.json
  - Run4 -> data/raw/Farm/Run4/labels_config.json
  - Run3 -> data/raw/Farm/Run3/labels_config.json
  - Run1 -> data/raw/Farm/Run1/labels_config.json

Run1:
  Label config: data/raw/Farm/Run1/labels_config.json
  Labels: grass, dirt_track, soil
  Samples: 52204

Run2:
  Label config: data/raw/Farm/Run2/labels_config.json
  Labels: grass, dirt_track, soil
  Samples: 90033

Run3:
  Label config: data/raw/Farm/Run3/labels_config.json
  Labels: grass, dirt_track, soil
  Samples: 80944

Run4:
  Label config: da

## Apply labels per run

For each run, look up its label config and assign a terrain class to every timestamp inside a labelled interval.


In [4]:
# Apply labels to each run's merged data
labeled_runs = {}

for run_id, run_info in run_data_dict.items():
    merged_run_df = run_info['data']
    label_configs = run_info['label_configs']
    
    # Apply labels using the 't' (Unix timestamp) column
    labeled_df = label_dataframe(merged_run_df, label_configs, time_column='t')
    labeled_runs[run_id] = labeled_df
    
    # Show label distribution
    label_counts = labeled_df['label'].value_counts(dropna=False)
    print(f"\n{run_id.upper()} - Label Distribution:")
    print(f"  Total samples: {len(labeled_df)}")
    for label, count in label_counts.items():
        percentage = (count / len(labeled_df)) * 100
        label_name = label if pd.notna(label) else "unlabeled"
        print(f"  {label_name}: {count} samples ({percentage:.1f}%)")



RUN1 - Label Distribution:
  Total samples: 52204
  dirt_track: 16200 samples (31.0%)
  unlabeled: 15304 samples (29.3%)
  grass: 11400 samples (21.8%)
  soil: 9300 samples (17.8%)

RUN2 - Label Distribution:
  Total samples: 90033
  unlabeled: 34533 samples (38.4%)
  grass: 21400 samples (23.8%)
  dirt_track: 21000 samples (23.3%)
  soil: 13100 samples (14.6%)

RUN3 - Label Distribution:
  Total samples: 80944
  grass: 29300 samples (36.2%)
  soil: 20300 samples (25.1%)
  dirt_track: 16600 samples (20.5%)
  unlabeled: 14744 samples (18.2%)

RUN4 - Label Distribution:
  Total samples: 80700
  dirt_track: 30800 samples (38.2%)
  soil: 24100 samples (29.9%)
  grass: 19300 samples (23.9%)
  unlabeled: 6500 samples (8.1%)

RUN5 - Label Distribution:
  Total samples: 163432
  unlabeled: 109232 samples (66.8%)
  grass: 24800 samples (15.2%)
  soil: 15800 samples (9.7%)
  dirt_track: 13600 samples (8.3%)


## Validate label transitions

Spot-check the first few class transitions per run to confirm boundary alignment.


In [5]:
# Validate label transitions for each run
for run_id, labeled_df in labeled_runs.items():
    print(f"\n{'='*60}")
    print(f"Validation: {run_id}")
    print('='*60)
    
    # Validate transitions
    validation = validate_label_transitions(labeled_df, max_transitions=3)
    
    print(f"Total transitions: {validation['total_transitions']}")
    for transition in validation['transition_samples']:
        print(f"\nTransition {transition['transition_num']} (row {transition['row_idx']}):")
        print(transition['samples'])



Validation: Run1
Total transitions: 15308

Transition 1 (row 1):
              t label
0  1.776765e+09   NaN
1  1.776765e+09   NaN
2  1.776765e+09   NaN
3  1.776765e+09   NaN

Transition 2 (row 2):
              t label
0  1.776765e+09   NaN
1  1.776765e+09   NaN
2  1.776765e+09   NaN
3  1.776765e+09   NaN
4  1.776765e+09   NaN

Transition 3 (row 3):
              t label
1  1.776765e+09   NaN
2  1.776765e+09   NaN
3  1.776765e+09   NaN
4  1.776765e+09   NaN
5  1.776765e+09   NaN

Validation: Run2
Total transitions: 34540

Transition 1 (row 1):
              t label
0  1.776765e+09   NaN
1  1.776765e+09   NaN
2  1.776765e+09   NaN
3  1.776765e+09   NaN

Transition 2 (row 2):
              t label
0  1.776765e+09   NaN
1  1.776765e+09   NaN
2  1.776765e+09   NaN
3  1.776765e+09   NaN
4  1.776765e+09   NaN

Transition 3 (row 3):
              t label
1  1.776765e+09   NaN
2  1.776765e+09   NaN
3  1.776765e+09   NaN
4  1.776765e+09   NaN
5  1.776765e+09   NaN

Validation: Run3
Total tran

## Persist the labelled dataset

Concatenate every run and write a single CSV to `data/interim/labeled/`.


In [6]:
# Save all labeled runs into a single labeled dataset
output_root = PROJECT_ROOT / "data" / "interim" / "labeled"
output_root.mkdir(parents=True, exist_ok=True)

# Concatenate all labeled runs
all_labeled_data = []
for run_id, labeled_df in labeled_runs.items():
    all_labeled_data.append(labeled_df)

labeled_dataset = pd.concat(all_labeled_data, ignore_index=True)

# Save to single CSV file
labeled_output_path = output_root / output_file
labeled_dataset.to_csv(labeled_output_path, index=False)

print(f"\nSaved labeled dataset to: {labeled_output_path}")
print(f"Total shape: {labeled_dataset.shape}")
print(f"Columns: {list(labeled_dataset.columns)}")
print(f"\nLabel distribution:")
label_counts = labeled_dataset['label'].value_counts(dropna=False)
for label, count in label_counts.items():
    percentage = (count / len(labeled_dataset)) * 100
    label_name = label if pd.notna(label) else "unlabeled"
    print(f"  {label_name}: {count} samples ({percentage:.1f}%)")

# Display summary per run
print(f"\nSamples per run:")
for run_id in labeled_dataset['run_id'].unique():
    n_samples = len(labeled_dataset[labeled_dataset['run_id'] == run_id])
    print(f"  {run_id}: {n_samples} samples")

# Display first few rows
labeled_dataset.head()



Saved labeled dataset to: /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/data/interim/labeled/labeled_dataset_Farm.csv
Total shape: (467313, 12)
Columns: ['t', 't_rel', 'run_id', 'ax', 'ay', 'az', 'gx', 'gy', 'gz', 'v1', 'v2', 'label']

Label distribution:
  unlabeled: 180313 samples (38.6%)
  grass: 106200 samples (22.7%)
  dirt_track: 98200 samples (21.0%)
  soil: 82600 samples (17.7%)

Samples per run:
  Run1: 52204 samples
  Run2: 90033 samples
  Run3: 80944 samples
  Run4: 80700 samples
  Run5: 163432 samples


,t,t_rel,run_id,ax,ay,az,gx,gy,gz,v1,v2,label
0,1.776765e+09,0.00,Run1,0.045100,-0.999500,0.006000,-3.615400,-9.824300,-1.360500,-0.0,0.0,NaN
1,1.776765e+09,0.01,Run1,0.045043,-1.000359,0.009035,-3.583422,-9.765124,-1.556052,0.0,0.0,NaN
2,1.776765e+09,0.02,Run1,0.043609,-1.000624,0.010435,-3.530044,-9.849917,-1.683888,0.0,0.0,NaN
3,1.776765e+09,0.03,Run1,0.042780,-0.999864,0.008805,-3.440862,-9.907412,-1.692812,0.0,0.0,NaN
4,1.776765e+09,0.04,Run1,0.048900,-0.999263,0.008000,-3.336847,-9.707005,-1.661626,0.0,0.0,NaN


## Combined labelled sensor plots

Per-run 3×2 panel (acc/gyro/odo axes + magnitudes) with label spans shaded by class.


In [7]:
# Generate and save combined labeled plots for all runs
import matplotlib.pyplot as plt

plots_output_root = PROJECT_ROOT / "reports" / "labeled"

for run_id, labeled_df in labeled_runs.items():
    # Check if combined plot already exists
    run_plot_dir = plots_output_root / run_id
    combined_plot_path = run_plot_dir / "labeled_sensors_plot.png"
    
    if combined_plot_path.exists():
        print(f"\n⏭ Skipping {run_id}: combined plot already exists")
        print(f"  {combined_plot_path.relative_to(PROJECT_ROOT)}")
        continue
    
    # Split into sensor DataFrames for plotting
    acc_labeled = labeled_df[['t', 't_rel', 'ax', 'ay', 'az', 'label']].copy()
    gyro_labeled = labeled_df[['t', 't_rel', 'gx', 'gy', 'gz', 'label']].copy()
    odo_labeled = labeled_df[['t', 't_rel', 'v1', 'v2', 'label']].copy()
    
    # Create combined labeled plot
    fig = plot_labeled_sensors(
        acc=acc_labeled,
        gyro=gyro_labeled,
        odo=odo_labeled,
        tcol='t_rel',
        show=False
    )
    
    # Save plot to reports/labeled/{run_id}/
    plot_path = save_labeled_plot(
        fig,
        run_id=run_id,
        output_root=plots_output_root,
        filename="labeled_sensors_plot.png",
        dpi=150
    )
    
    plt.close(fig)  # Close figure to free memory
    
    print(f"\n✓ Saved combined plot for {run_id}:")
    print(f"  {plot_path.relative_to(PROJECT_ROOT)}")



⏭ Skipping Run1: combined plot already exists
  reports/labeled/Run1/labeled_sensors_plot.png

⏭ Skipping Run2: combined plot already exists
  reports/labeled/Run2/labeled_sensors_plot.png

✓ Saved combined plot for Run3:
  reports/labeled/Run3/labeled_sensors_plot.png

✓ Saved combined plot for Run4:
  reports/labeled/Run4/labeled_sensors_plot.png

✓ Saved combined plot for Run5:
  reports/labeled/Run5/labeled_sensors_plot.png


## Per-sensor labelled plots

Separate accelerometer, gyroscope, and odometry breakdowns saved under `reports/labeled/`.


In [8]:
# Generate and save individual sensor plots (accelerometer, gyroscope, odometry)
import matplotlib.pyplot as plt

plots_output_root = PROJECT_ROOT / "reports" / "labeled"

for run_id, labeled_df in labeled_runs.items():
    # Check if all individual plots already exist
    run_plot_dir = plots_output_root / run_id
    acc_plot_path = run_plot_dir / "labeled_acc_plot.png"
    gyro_plot_path = run_plot_dir / "labeled_gyro_plot.png"
    odo_plot_path = run_plot_dir / "labeled_odo_plot.png"
    
    all_exist = acc_plot_path.exists() and gyro_plot_path.exists() and odo_plot_path.exists()
    
    if all_exist:
        print(f"\n⏭ Skipping {run_id}: all individual plots already exist")
        continue
    
    print(f"\n{'='*60}")
    print(f"Saving individual plots for {run_id}")
    print('='*60)
    
    # Split into sensor DataFrames for plotting
    acc_labeled = labeled_df[['t', 't_rel', 'ax', 'ay', 'az', 'label']].copy()
    gyro_labeled = labeled_df[['t', 't_rel', 'gx', 'gy', 'gz', 'label']].copy()
    odo_labeled = labeled_df[['t', 't_rel', 'v1', 'v2', 'label']].copy()
    
    # Plot and save accelerometer (if not exists)
    if not acc_plot_path.exists():
        fig_acc = plot_labeled_acc(
            acc=acc_labeled,
            tcol='t_rel',
            show=False
        )
        plot_path_acc = save_labeled_plot(
            fig_acc,
            run_id=run_id,
            output_root=plots_output_root,
            filename="labeled_acc_plot.png",
            dpi=150
        )
        plt.close(fig_acc)
        print(f"✓ Saved accelerometer plot:")
        print(f"  {plot_path_acc.relative_to(PROJECT_ROOT)}")
    else:
        print(f"⏭ Accelerometer plot exists, skipping")
    
    # Plot and save gyroscope (if not exists)
    if not gyro_plot_path.exists():
        fig_gyro = plot_labeled_gyro(
            gyro=gyro_labeled,
            tcol='t_rel',
            show=False
        )
        plot_path_gyro = save_labeled_plot(
            fig_gyro,
            run_id=run_id,
            output_root=plots_output_root,
            filename="labeled_gyro_plot.png",
            dpi=150
        )
        plt.close(fig_gyro)
        print(f"✓ Saved gyroscope plot:")
        print(f"  {plot_path_gyro.relative_to(PROJECT_ROOT)}")
    else:
        print(f"⏭ Gyroscope plot exists, skipping")
    
    # Plot and save odometry (if not exists)
    if not odo_plot_path.exists():
        fig_odo = plot_labeled_odo(
            odo=odo_labeled,
            tcol='t_rel',
            show=False
        )
        plot_path_odo = save_labeled_plot(
            fig_odo,
            run_id=run_id,
            output_root=plots_output_root,
            filename="labeled_odo_plot.png",
            dpi=150
        )
        plt.close(fig_odo)
        print(f"✓ Saved odometry plot:")
        print(f"  {plot_path_odo.relative_to(PROJECT_ROOT)}")
    else:
        print(f"⏭ Odometry plot exists, skipping")



⏭ Skipping Run1: all individual plots already exist

⏭ Skipping Run2: all individual plots already exist

Saving individual plots for Run3
✓ Saved accelerometer plot:
  reports/labeled/Run3/labeled_acc_plot.png
✓ Saved gyroscope plot:
  reports/labeled/Run3/labeled_gyro_plot.png
✓ Saved odometry plot:
  reports/labeled/Run3/labeled_odo_plot.png

Saving individual plots for Run4
✓ Saved accelerometer plot:
  reports/labeled/Run4/labeled_acc_plot.png
✓ Saved gyroscope plot:
  reports/labeled/Run4/labeled_gyro_plot.png
✓ Saved odometry plot:
  reports/labeled/Run4/labeled_odo_plot.png

Saving individual plots for Run5
✓ Saved accelerometer plot:
  reports/labeled/Run5/labeled_acc_plot.png
✓ Saved gyroscope plot:
  reports/labeled/Run5/labeled_gyro_plot.png
✓ Saved odometry plot:
  reports/labeled/Run5/labeled_odo_plot.png
